# NEXUS continuous — run 1 or all environments

Trains the state-based NEXUS suite; select one env or all 6, get per-env curves + eval table.
The RGB (pixel) extension is CartpoleBalance-only — see `rgb_distillation_colab.ipynb`.

Use a **GPU (T4)** runtime.

In [ ]:
import os, subprocess
if subprocess.run('nvidia-smi').returncode:
    print('WARNING: no GPU — training will be slow.')
os.environ['XLA_FLAGS'] = os.environ.get('XLA_FLAGS','') + ' --xla_gpu_triton_gemm_any=True'
os.environ['JAX_DEFAULT_MATMUL_PRECISION'] = 'highest'

import os
REPO_URL = 'https://github.com/Tornadosky/nexus_project.git'
BRANCH   = 'rosela'
%cd /content
if not os.path.exists('/content/nexus_project'):
    !git clone {REPO_URL}
%cd /content/nexus_project
!git checkout {BRANCH} && git pull -q origin {BRANCH} || echo '(using local checkout)'
%cd /content/nexus_project/nexus_continuous_control
print('cwd:', os.getcwd())

In [ ]:
# State training uses impl='jax' (no warp), so no MJWarp issues here.
!pip install -q -e ".[dev,analysis,playground]"
import jax; print('jax', jax.__version__, jax.devices())

## Select environments + budget
`RUN_ALL=True` runs all 6. `MODE='demo'` = short budget (curves move, not converged); `MODE='full'` = config budget (long: ~10-40 min/env on a T4).

In [ ]:
CONFIGS = {
    'CartpoleBalance':        'cartpole_balance_nesy',
    'CheetahRun':             'cheetah_run_nesy',
    'WalkerWalk':             'walker_walk_nesy',
    'HopperHop':              'hopper_hop_nesy',
    'PandaPickCube':          'panda_pick_cube_nesy',
    'Go1JoystickFlatTerrain': 'go1_joystick_nesy',
}
RUN_ALL = False
ENVS    = list(CONFIGS) if RUN_ALL else ['CartpoleBalance']
MODE    = 'demo'          # 'demo' or 'full'
print('run:', ENVS, '| mode:', MODE)

In [ ]:
import os
os.makedirs('/content/runs', exist_ok=True)
DEMO = '--override TOTAL_TIMESTEPS=1048576 --override NUM_SEEDS=1'
for env in ENVS:
    ov = DEMO if MODE == 'demo' else ''
    print(f'\n===== {env} =====')
    !python -m nexus_continuous.scripts.train_nexus_playground \
      --config configs/{CONFIGS[env]}.yaml {ov} \
      --save /content/runs/{env}.pkl --no-wandb 2>&1 | tail -22

## Results — curves + eval table

In [ ]:
import os, pickle, numpy as np, matplotlib.pyplot as plt
runs = {e: f'/content/runs/{e}.pkl' for e in ENVS if os.path.exists(f'/content/runs/{e}.pkl')}
if not runs: raise SystemExit('No checkpoints — did training run?')
fig, axes = plt.subplots(1, len(runs), figsize=(4.2*len(runs), 3.2), squeeze=False)
print(f'{"environment":24s} {"eval return":>12s} {"success":>10s}')
for ax, (env, path) in zip(axes[0], runs.items()):
    ck = pickle.load(open(path, 'rb')); m = ck['metrics']; ev = ck.get('eval_metrics', {})
    key = 'rollout/episode_return' if 'rollout/episode_return' in m else 'returns/env_reward_mean'
    ax.plot(np.asarray(m[key]).reshape(-1)); ax.set_title(env, fontsize=9)
    ax.set_xlabel('update'); ax.set_ylabel('return'); ax.grid(alpha=0.3)
    ret = float(np.asarray(ev.get('episode_return_mean', np.nan)))
    succ = float(np.asarray(ev['primary_success_rate'])) if 'primary_success_rate' in ev else float('nan')
    print(f'{env:24s} {ret:12.2f} {succ:10.3f}')
plt.tight_layout(); plt.show()